In [ ]:
# Сохраняем y_true немного кривым способом

In [ ]:
# import glob
# import os
# import torch
# import tqdm
# import pickle

# from hydra import compose, initialize
# from cmip6_cmip6_dl_base_model import main

# CONFIG_NAME_TRAIN = "cmip6-cmip6"
# CONFIG_NAME_INFERENCE = "cmip6-cmip6_factor4"

# with initialize(version_base=None, config_path="configs/inference"):
#     cfg_inference = compose(config_name=CONFIG_NAME_INFERENCE)

# with initialize(version_base=None, config_path="configs/train"):
#     cfg_train = compose(config_name=CONFIG_NAME_TRAIN)

# i = 0,
# seed = 42
# model = 'unet'

# version=0
# ckpt_dir = glob.glob(os.path.join(cfg_inference.path, f"{model}_multi_{cfg_inference[model][0].upsampling}_{seed}/logs/version_{version}/checkpoints/"+"epoch_*.*"))
# cfg_train["model"].upsampling = cfg_inference[model][0].upsampling
# cfg_train.model.architecture = model
# cfg_train.training.checkpoint = ckpt_dir[-1]

# model_module, dm = main(cfg_train);
# model_module.eval().to('cuda:5')
# dm.setup()

# y_true_train = []
# with torch.no_grad():
#     for batch in dm.predict_dataloader():
#         _, y, _, _ = batch
#         y_true_train.append(y)

# y_true_val = []
# with torch.no_grad():
#     for batch in dm.val_dataloader():
#         _, y, _, _ = batch
#         y_true_val.append(y)

# y_true_test = []
# with torch.no_grad():
#     for batch in dm.test_dataloader():
#         _, y, _, _ = batch
#         y_true_test.append(y)

In [ ]:
# print('saving train')
# with open('y_true_train_ens_data_factor_4.pkl', 'wb') as f:
#     pickle.dump(torch.cat(y_true_train), f)
# print('done')

# print('saving val')
# with open('y_true_val_ens_data_factor_4.pkl', 'wb') as f:
#     pickle.dump(torch.cat(y_true_val), f)
# print('done')

# print('saving test')
# with open('y_true_test_ens_data_factor_4.pkl', 'wb') as f:
#     pickle.dump(torch.cat(y_true_test), f)
# print('done')

# Обучаем МЛП

In [1]:
import pickle
import torch

from torchmetrics.regression import MeanSquaredError
from torchmetrics.functional.image import peak_signal_noise_ratio
from torchmetrics.functional.image import structural_similarity_index_measure

## Скачиваем трейн

In [2]:
with open('/app/train_ens_data_factor_4_unet_42.pkl', 'rb') as f:
    train_ens_data_factor_4_unet_42 = pickle.load(f)

with open('/app/train_ens_data_factor_4_vit_42.pkl', 'rb') as f:
    train_ens_data_factor_4_vit_42 = pickle.load(f)

with open('/app/train_ens_data_factor_4_hat_42.pkl', 'rb') as f:
    train_ens_data_factor_4_hat_42 = pickle.load(f)

with open('/app/train_ens_data_factor_4_diffusion_42.pkl', 'rb') as f:
    train_ens_data_factor_4_diffusion_42 = pickle.load(f)


In [3]:
with open('data/processed/cmip6-cmip6/preds_factor2/y_true_train_ens_data.pkl', 'rb') as f:
    train_y_true = pickle.load(f)

## Скачиваем валидацию

In [4]:
with open('/app/val_ens_data_factor_4_unet_42.pkl', 'rb') as f:
    val_ens_data_factor_4_unet_42 = pickle.load(f)

with open('/app/val_ens_data_factor_4_vit_42.pkl', 'rb') as f:
    val_ens_data_factor_4_vit_42 = pickle.load(f)

with open('/app/val_ens_data_factor_4_hat_42.pkl', 'rb') as f:
    val_ens_data_factor_4_hat_42 = pickle.load(f)

with open('/app/val_ens_data_factor_4_diffusion_42.pkl', 'rb') as f:
    val_ens_data_factor_4_diffusion_42 = pickle.load(f)


In [5]:
with open('/app/y_true_val_ens_data_factor_4.pkl', 'rb') as f:
    val_y_true = pickle.load(f)

## Скачиваем тест

In [6]:
with open('/app/test_ens_data_factor_4_unet_42.pkl', 'rb') as f:
    test_ens_data_factor_4_unet_42 = pickle.load(f)

with open('/app/test_ens_data_factor_4_vit_42.pkl', 'rb') as f:
    test_ens_data_factor_4_vit_42 = pickle.load(f)

with open('/app/test_ens_data_factor_4_hat_42.pkl', 'rb') as f:
    test_ens_data_factor_4_hat_42 = pickle.load(f)

with open('/app/test_ens_data_factor_4_diffusion_42.pkl', 'rb') as f:
    test_ens_data_factor_4_diffusion_42 = pickle.load(f)


In [7]:
with open('/app/y_true_test_ens_data_factor_4.pkl', 'rb') as f:
    test_y_true = pickle.load(f)

## Создаем единый тензор

In [8]:
torch.stack([
    val_ens_data_factor_4_unet_42['unet_42'],
    val_ens_data_factor_4_vit_42['vit_42'],
    val_ens_data_factor_4_hat_42['hat_42'],
    val_ens_data_factor_4_diffusion_42['diffusion_42'],
], dim=1).shape

torch.Size([292, 4, 4, 192, 384])

In [9]:
train_data = torch.stack([
    train_ens_data_factor_4_unet_42['unet_42'],
    train_ens_data_factor_4_vit_42['vit_42'],
    train_ens_data_factor_4_hat_42['hat_42'],
    train_ens_data_factor_4_diffusion_42['diffusion_42'],
], dim=1)

val_data = torch.stack([
    val_ens_data_factor_4_unet_42['unet_42'],
    val_ens_data_factor_4_vit_42['vit_42'],
    val_ens_data_factor_4_hat_42['hat_42'],
    val_ens_data_factor_4_diffusion_42['diffusion_42'],
], dim=1)

test_data = torch.stack([
    test_ens_data_factor_4_unet_42['unet_42'],
    test_ens_data_factor_4_vit_42['vit_42'],
    test_ens_data_factor_4_hat_42['hat_42'],
    test_ens_data_factor_4_diffusion_42['diffusion_42'],
], dim=1)


# Код реализации MLP

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import tqdm
from typing import Tuple, List
import numpy as np

class ClimateMLP(nn.Module):
    """Улучшенная версия с несколькими слоями"""
    def __init__(self, n_models: int = 4, n_channels: int = 4, hidden_channels: int = 16):
        super().__init__()
        
        self.network = nn.Sequential(
            # Первый слой: объединение моделей
            nn.Conv2d(n_models * n_channels, hidden_channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            
            # Второй слой: дополнительная обработка
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            
            # Третий слой: дополнительная обработка
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            
            # Финальный слой: проекция к нужному числу каналов
            nn.Conv2d(hidden_channels, n_channels, kernel_size=1)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        height, width = x.shape[-2], x.shape[-1]
        x_reshaped = x.view(batch_size, -1, height, width)
        return self.network(x_reshaped)


def train_mlp_ensemble(X_train_tensor,
                       X_val_tensor,
                       y_train_tensor,
                       y_val_tensor,
                       device,
                       epochs=100,
                       batch_size=8,
                       learning_rate=0.001,
                       weight_decay=1e-4):
    """
    Обучение MLP ансамбля с улучшенным оптимизатором и стратегией LR scheduling
    
    Args:
        X: входные данные формы (n_samples, n_models, 4, 192, 384)
        y: целевые данные формы (n_samples, 4, 192, 384)
        n_models: количество моделей в ансамбле
        epochs: количество эпох
        batch_size: размер батча
        learning_rate: скорость обучения
        weight_decay: коэффициент L2 регуляризации
    """
    
    # Создание DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Инициализация модели
    model = ClimateMLP().to(device)
    
    # Функция потерь и оптимизатор - используем AdamW вместо Adam
    criterion = nn.MSELoss()
    
    # AdamW с weight decay для лучшей регуляризации
    optimizer = optim.AdamW(model.parameters(), 
                          lr=learning_rate, 
                          weight_decay=weight_decay,
                          betas=(0.9, 0.999))
    
    # Комбинированная стратегия уменьшения learning rate
    # 1. Cosine Annealing с теплым стартом
    scheduler_cosine = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, 
        T_0=20,           # Количество эпох до первого restart
        T_mult=2,         # Умножение T_0 после каждого restart
        eta_min=1e-6      # Минимальный learning rate
    )
    
    # 2. ReduceLROnPlateau как fallback
    scheduler_plateau = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min',
        patience=8,       # Увеличиваем patience
        factor=0.5, 
        verbose=True,
        min_lr=1e-7
    )
    
    # Для сохранения истории обучения
    train_losses = []
    val_losses = []
    learning_rates = []
    
    print("Начинаем обучение MLP ансамбля...")
    print(f"Устройство: {device}")
    print(f"Размер тренировочной выборки: {len(X_train_tensor)}")
    print(f"Размер валидационной выборки: {len(X_val_tensor)}")
    print(f"Оптимизатор: AdamW с weight_decay={weight_decay}")
    print(f"Стратегия LR: CosineAnnealingWarmRestarts + ReduceLROnPlateau")
    
    best_val_loss = float('inf')
    patience_counter = 0
    patience = 15  # Ранняя остановка
    
    for epoch in tqdm.tqdm(range(epochs)):
        # Тренировка
        model.train()
        train_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            
            # Gradient clipping для стабильности
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item() * batch_X.size(0)
        
        # Валидация
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item() * batch_X.size(0)
        
        # Средние потери
        train_loss = train_loss / len(train_loader.dataset)
        val_loss = val_loss / len(val_loader.dataset)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        learning_rates.append(optimizer.param_groups[0]['lr'])
        
        # Обновление learning rate
        scheduler_cosine.step()
        scheduler_plateau.step(val_loss)
        
        # Проверка на лучшую модель
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Сохраняем лучшие веса
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        # Ранняя остановка
        if patience_counter >= patience:
            print(f"Ранняя остановка на эпохе {epoch+1}")
            # Загружаем лучшие веса
            model.load_state_dict(best_model_state)
            break
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], '
                  f'Train Loss: {train_loss:.6f}, '
                  f'Val Loss: {val_loss:.6f}, '
                  f'LR: {optimizer.param_groups[0]["lr"]:.2e}, '
                  f'Best Val: {best_val_loss:.6f}')
    
    # График обучения
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # График потерь
    ax1.plot(train_losses, label='Train Loss')
    ax1.plot(val_losses, label='Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training History')
    ax1.legend()
    ax1.set_yscale('log')
    
    # График learning rate
    ax2.plot(learning_rates, label='Learning Rate', color='red')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Learning Rate')
    ax2.set_title('Learning Rate Schedule')
    ax2.set_yscale('log')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    return model, train_losses, val_losses, learning_rates

In [ ]:
# Обучение модели
model, train_losses, val_losses, learning_rates = train_mlp_ensemble(
    train_data,
    val_data,
    train_y_true,
    val_y_true, 
    device='cuda:0',
    epochs=3,
    batch_size=4,
    learning_rate=0.001,
    weight_decay=1e-4
)


PATH = "my_model_weights.pth"
torch.save(model.state_dict(), PATH)
                       

/opt/conda/envs/bias_correction/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Начинаем обучение MLP ансамбля...
Устройство: cuda:1
Размер тренировочной выборки: 4672
Размер валидационной выборки: 292
Оптимизатор: AdamW с weight_decay=0.0001
Стратегия LR: CosineAnnealingWarmRestarts + ReduceLROnPlateau


  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:15<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
train_data.shape

In [ ]:
train_y_true.shape